# Kernel PCA on two ring roads

A delivery company logs where its vans stop, as east and north coordinates in kilometers from the city center, and the stops lie on two ring roads. The company wants one number per stop that says which road it was on, and no straight axis gives one, so linear PCA cannot. Here you build kernel PCA from the lesson's formulas, a Gaussian similarity between every pair of stops, a kernel matrix centered in feature space and its top eigenvectors, and find the window of bandwidths in which one component reads the road.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It builds 400 stops on the two ring roads with a seeded random generator, and it carries over linear PCA from the PCA notebook as `pca_scores`, since that part is not the subject here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

# 400 van stops around a city, as (east, north) in kilometers from the center.
# 200 are on the inner ring road, 3 km out, and 200 on the outer one, 10 km
# out, each with about 0.4 km of scatter. `road` says which: 0 inner, 1 outer.
rng = np.random.default_rng(0)
n_each = 200
angle = rng.uniform(0, 2 * np.pi, 2 * n_each)
radius = np.repeat([3.0, 10.0], n_each) + rng.normal(0, 0.4, 2 * n_each)
X = np.column_stack([radius * np.cos(angle), radius * np.sin(angle)])
road = np.repeat([0, 1], n_each)
print("stops:", X.shape, "  on the inner road:", (road == 0).sum(), "  on the outer road:", (road == 1).sum())


def pca_scores(X, k):
    """Scores on the first k principal components, as built in the PCA notebook."""
    Xc = X - X.mean(axis=0)
    vals, vecs = np.linalg.eigh(Xc.T @ Xc / (len(X) - 1))
    order = np.argsort(vals)[::-1]
    return Xc @ vecs[:, order][:, :k]


plt.figure(figsize=(4.5, 4.5))
plt.scatter(X[:, 0], X[:, 1], c=road, cmap="coolwarm", s=10)
plt.xlabel("east (km)")
plt.ylabel("north (km)")
plt.gca().set_aspect("equal")
plt.title("van stops on the two ring roads")
plt.show()

## Part 1 (coding task): Every distance at once

Every method in the lesson starts from the distances between pairs of points. Write `squared_distances(A, B)`, which returns the matrix whose entry $(i, j)$ is the squared Euclidean distance between row $i$ of `A` and row $j$ of `B`. Broadcasting does it without a loop: `A[:, None, :] - B[None, :, :]` is a three-dimensional array of all the differences, and squaring it and summing over the last axis gives the matrix. The Gaussian kernel wants squared distances, so there is no square root to take.

In [ ]:
def squared_distances(A, B):
    """Matrix of squared Euclidean distances between the rows of A and the rows of B."""
    # TODO
    raise NotImplementedError

In [ ]:
P = np.array([[0.0, 0.0], [3.0, 4.0], [1.0, 1.0]])
D_small = squared_distances(P, P)
print(D_small)
assert D_small.shape == (3, 3), f"three points against three give a 3 x 3 matrix, got {D_small.shape}"
assert np.allclose(D_small, [[0, 25, 2], [25, 0, 13], [2, 13, 0]]), "(0, 0) to (3, 4) is 25 squared, (0, 0) to (1, 1) is 2"
assert squared_distances(X[:5], X[:3]).shape == (5, 3), "A and B may have different numbers of rows"
assert np.isclose(squared_distances(X[:5], X[:3])[4, 2], ((X[4] - X[2]) ** 2).sum()), "entry (i, j) is the squared distance from row i of A to row j of B"

D = squared_distances(X, X)
assert D.shape == (400, 400) and np.allclose(D, D.T), "400 stops give a symmetric 400 x 400 matrix"
assert np.allclose(np.diag(D), 0), "every stop is at distance 0 from itself"
print("largest squared distance between two stops:", round(float(D.max()), 2), "km squared")
print("Part 1 passed")

## Part 2 (coding task): Can one number tell the roads apart?

The company wants a single number per stop, and a number works when one threshold on it separates the roads: everything below the threshold is one road and everything above is the other. Write `threshold_accuracy(values, labels)`, the accuracy of the best threshold on `values`. Try each value as the threshold, predict road 1 for every stop above it, and compute the accuracy. The flipped prediction, road 1 below the threshold, counts too, and its accuracy is one minus the first. Return the best over all of them, as a float.

Then the check measures linear PCA with it. The first principal component of the stops is a straight axis through the rings. A threshold at the edge of the inner ring catches the outer stops at one end of that axis and nothing else, so no straight axis does much better than 0.7, while the distance from the center scores 1.0.

In [ ]:
def threshold_accuracy(values, labels):
    """Accuracy of the best single threshold on `values`, in either direction."""
    # TODO
    raise NotImplementedError

In [ ]:
assert threshold_accuracy(np.array([1.0, 2.0, 3.0, 4.0]), np.array([0, 0, 1, 1])) == 1.0, "a cut between 2 and 3 is perfect"
assert threshold_accuracy(np.array([1.0, 2.0, 3.0, 4.0]), np.array([1, 1, 0, 0])) == 1.0, "the flipped direction has to count too"
assert np.isclose(threshold_accuracy(np.array([1.0, 2.0, 3.0, 4.0]), np.array([0, 1, 0, 1])), 0.75), "the best cut on 0, 1, 0, 1 gets three of four"

dist_from_center = np.sqrt((X ** 2).sum(axis=1))
print("distance from the center:     ", threshold_accuracy(dist_from_center, road))
assert threshold_accuracy(dist_from_center, road) == 1.0, "the distance from the center separates the roads perfectly, so the measure should say 1.0"

linear = pca_scores(X, 2)
acc_pc1 = threshold_accuracy(linear[:, 0], road)
print("first principal component:    ", acc_pc1)
print("second principal component:   ", threshold_accuracy(linear[:, 1], road))
assert acc_pc1 < 0.8, f"a straight axis through two rings cannot separate them, so this should be near 0.7, got {acc_pc1}"

plt.figure(figsize=(6, 3))
for r, name in [(0, "inner road"), (1, "outer road")]:
    plt.hist(linear[road == r, 0], bins=30, alpha=0.6, label=name)
plt.xlabel("score on the first principal component")
plt.legend()
plt.show()
print("Part 2 passed")

## Part 3 (coding task): Similarities with a bandwidth

The lesson turns distances into similarities with a Gaussian, and t-SNE normalizes one point's similarities into a probability distribution over its neighbors. Write `neighbor_probabilities(d_sq, sigma)`, which takes a point's squared distances to the other points and returns $p_{j \mid i} \propto \exp(-d_j^2 / 2\sigma^2)$, normalized to sum to 1. Then write `perplexity(p)`, which is $2^{H}$ with $H = -\sum_j p_j \log_2 p_j$, the effective number of neighbors. Skip the entries of `p` that are exactly 0, since $0 \log 0$ counts as 0.

The lesson's example is a point with neighbors at distances 1 and 2. With $\sigma = 1$ the probabilities are 0.8176 and 0.1824 and the perplexity is 1.608. A wider Gaussian spreads the probability and raises the perplexity, and a narrower one lowers it.

In [ ]:
def neighbor_probabilities(d_sq, sigma):
    """Gaussian similarities to the other points, normalized to a probability distribution."""
    # TODO
    raise NotImplementedError


def perplexity(p):
    """2 to the entropy of p in bits: the effective number of neighbors."""
    # TODO
    raise NotImplementedError

In [ ]:
p = neighbor_probabilities(np.array([1.0, 4.0]), 1.0)      # neighbors at distances 1 and 2, squared
print("p =", p, " perplexity", round(perplexity(p), 4))
assert np.isclose(p.sum(), 1), "the probabilities should sum to 1"
assert np.allclose(p, [0.8176, 0.1824], atol=1e-4), f"with sigma = 1 the lesson gets (0.8176, 0.1824), got {p}"
assert np.isclose(perplexity(p), 1.608, atol=1e-3), f"the perplexity should be 1.608, got {perplexity(p)}"
assert np.isclose(perplexity(neighbor_probabilities(np.array([1.0, 4.0]), 2.0)), 1.966, atol=1e-3), "a wider Gaussian, sigma = 2, raises the perplexity to 1.966"
assert np.isclose(perplexity(neighbor_probabilities(np.array([1.0, 4.0]), 0.5)), 1.017, atol=1e-3), "a narrower one, sigma = 0.5, lowers it to 1.017"
assert np.isclose(perplexity(np.full(30, 1 / 30)), 30), "an even spread over 30 points has perplexity 30"
assert np.isclose(perplexity(np.array([1.0, 0.0])), 1), "all the probability on one point is perplexity 1, and 0 log 0 counts as 0"

others = np.arange(1, 400)                                  # stop 0 against every other stop
for sigma in (0.5, 2.0, 8.0):
    p0 = neighbor_probabilities(D[0, others], sigma)
    print(f"sigma = {sigma:>3} km: stop 0 has {perplexity(p0):7.2f} effective neighbors")
assert perplexity(neighbor_probabilities(D[0, others], 0.5)) < perplexity(neighbor_probabilities(D[0, others], 8.0)), "perplexity rises with sigma"
print("Part 3 passed")

## Part 4 (coding task): The kernel matrix, centered

Kernel PCA uses the same Gaussian as a kernel, $k(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \lVert \mathbf{x} - \mathbf{x}' \rVert^2)$ with $\gamma = 1 / 2\sigma^2$, and without the normalization. Write `rbf_kernel(D_sq, gamma)`, which turns a matrix of squared distances into the kernel matrix $K$. Then write `center_kernel(K)`, the lesson's centering in feature space: subtract each entry's column mean and row mean and add back the mean of the whole matrix, which is $\tilde K = K - \mathbf{1}K - K\mathbf{1} + \mathbf{1}K\mathbf{1}$ with $\mathbf{1}$ the matrix whose entries are all $1/n$. `K.mean(axis=0)` is the column means, `K.mean(axis=1, keepdims=True)` the row means as a column, and `K.mean()` the overall mean, and broadcasting lines them up.

The check uses a linear kernel, $K = X X^\top$, where centering the kernel has to agree with centering the data.

In [ ]:
def rbf_kernel(D_sq, gamma):
    """The Gaussian kernel exp(-gamma * squared distance), entry by entry."""
    # TODO
    raise NotImplementedError


def center_kernel(K):
    """K with its column means and row means subtracted and the overall mean added back."""
    # TODO
    raise NotImplementedError

In [ ]:
gamma = 0.03
K = rbf_kernel(D, gamma)
assert K.shape == (400, 400), "one entry per pair of stops"
assert np.allclose(np.diag(K), 1), "every stop is perfectly similar to itself"
assert np.allclose(K, K.T) and K.min() > 0 and K.max() <= 1, "similarities lie in (0, 1] and are symmetric"
assert np.isclose(K[0, 1], np.exp(-gamma * D[0, 1])), "entry (0, 1) should be exp(-gamma * D[0, 1])"

Kc = center_kernel(K)
assert Kc.shape == K.shape, "centering keeps the shape"
assert np.allclose(Kc.sum(axis=0), 0) and np.allclose(Kc.sum(axis=1), 0), "every row and column of a centered kernel sums to zero"
Xc = X - X.mean(axis=0)
assert np.allclose(center_kernel(X @ X.T), Xc @ Xc.T), "with a linear kernel, centering K has to match centering X"
print(f"mean similarity between two stops at gamma = {gamma}: {K.mean():.4f}")
print("Part 4 passed")

## Part 5 (coding task): The top eigenvectors of the kernel

Take a unit eigenvector $\mathbf{a}$ of $\tilde K$ with eigenvalue $\lambda$. The scores of the training points on that component are the entries of $\sqrt{\lambda}\,\mathbf{a}$. Write `kernel_pca(Kc, k)`, which returns `(lam, A, scores)`: the `k` largest eigenvalues of the centered kernel in descending order, the matching unit eigenvectors as the columns of `A`, and the $n \times k$ matrix of scores, each column of `A` scaled by the square root of its eigenvalue. `np.linalg.eigh` again returns the eigenvalues in ascending order.

With a linear kernel this has to give ordinary PCA's scores, up to the sign of each component, and its eigenvalues are $(n - 1)$ times PCA's. With the Gaussian kernel at $\gamma = 0.03$ the first component reads the road.

In [ ]:
def kernel_pca(Kc, k):
    """(lam, A, scores): the top k eigenpairs of the centered kernel and the training scores."""
    # TODO
    raise NotImplementedError

In [ ]:
lam_lin, A_lin, scores_lin = kernel_pca(center_kernel(X @ X.T), 2)
assert lam_lin.shape == (2,) and A_lin.shape == (400, 2) and scores_lin.shape == (400, 2), "two components: 2 eigenvalues, a 400 x 2 A and 400 x 2 scores"
assert lam_lin[0] >= lam_lin[1] > 0, "the largest eigenvalue comes first"
assert np.allclose(np.abs(A_lin.T @ A_lin), np.eye(2)), "the columns of A are unit vectors, perpendicular to each other"
assert np.allclose(np.abs(scores_lin), np.abs(pca_scores(X, 2))), "a linear kernel has to reproduce ordinary PCA, up to sign"
assert np.allclose(lam_lin, 399 * np.var(pca_scores(X, 2), axis=0, ddof=1)), "its eigenvalues are (n - 1) times PCA's"

lam, A, scores = kernel_pca(Kc, 2)
acc = [threshold_accuracy(scores[:, j], road) for j in range(2)]
print(f"gamma = {gamma}: threshold accuracy of kernel component 1: {acc[0]:.3f}, component 2: {acc[1]:.3f}")
assert acc[0] == 1.0, f"at gamma = 0.03 the first kernel component should separate the roads perfectly, got {acc[0]}"

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for r, name in [(0, "inner road"), (1, "outer road")]:
    axes[0].hist(scores[road == r, 0], bins=30, alpha=0.6, label=name)
    axes[1].scatter(scores[road == r, 0], scores[road == r, 1], s=8, label=name)
axes[0].set_xlabel("kernel component 1")
axes[0].legend()
axes[1].set_xlabel("kernel component 1")
axes[1].set_ylabel("kernel component 2")
plt.show()
print("Part 5 passed")

## Part 6 (coding task): The window of bandwidths

Put the pieces together. Write `kernel_pca_scores(X, gamma, k)`: squared distances, the Gaussian kernel, centering, and the top `k` scores, in one function. Then the check sweeps $\gamma$ and scores the first component with `threshold_accuracy`. Too wide a kernel makes every pair of stops look alike, and too narrow a one makes each stop resemble only itself.

The lesson found the window on rings of radius 0.3 and 1 to be about 2 to 5. These rings are ten times larger, and $\gamma$ multiplies a squared distance, so the window should sit near a hundredth of that.

In [ ]:
def kernel_pca_scores(X, gamma, k):
    """Kernel PCA scores of the rows of X with a Gaussian kernel of bandwidth gamma."""
    # TODO
    raise NotImplementedError

In [ ]:
assert np.allclose(np.abs(kernel_pca_scores(X, 0.03, 2)), np.abs(scores)), "at gamma = 0.03 this should reproduce Part 5"

gammas = [0.003, 0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.3, 1, 3]
sweep = []
for g in gammas:
    s = kernel_pca_scores(X, g, 2)
    assert s.shape == (400, 2), f"400 stops and two components give shape (400, 2), got {s.shape}"
    sweep.append(threshold_accuracy(s[:, 0], road))
    print(f"gamma {g:<6} first component separates the roads with accuracy {sweep[-1]:.3f}")
assert max(sweep) == 1.0, "some gamma separates the roads perfectly"
assert sweep[0] < 0.8 and sweep[-1] < 0.9, "the widest and the narrowest kernels both miss"

plt.figure(figsize=(5, 3))
plt.semilogx(gammas, sweep, "o-")
plt.xlabel("gamma")
plt.ylabel("accuracy of the best threshold")
plt.ylim(0.4, 1.05)
plt.show()
print("Part 6 passed")

## Part 7 (coding task): A stop the model has never seen

Kernel PCA defines a mapping, which is what makes it a feature transform and not only a picture. For a new point the lesson gives the score $\tilde{\mathbf{k}}^\top \mathbf{a} / \sqrt{\lambda}$, where $\tilde{\mathbf{k}}$ is its row of kernel values against the training points, centered with the training statistics: subtract the column means of the training $K$ and the row's own mean, and add back the mean of all of $K$. Write `transform_new(X_new, X_train, K_train, lam, A, gamma)`, which returns the scores of the rows of `X_new` on the components in `A`, one row per new point.

A training point fed back in must land on its own training score. The check then places 100 new stops, drawn with a different seed, by one threshold on the first component.

In [ ]:
def transform_new(X_new, X_train, K_train, lam, A, gamma):
    """Scores of new points on the components found from X_train."""
    # TODO
    raise NotImplementedError

In [ ]:
back = transform_new(X[:10], X, K, lam, A, gamma)
assert back.shape == (10, 2), f"ten stops and two components give shape (10, 2), got {back.shape}"
assert np.allclose(back, scores[:10]), "a training stop fed back in must land on its own training score"

rng_new = np.random.default_rng(1)
angle_new = rng_new.uniform(0, 2 * np.pi, 100)
radius_new = np.repeat([3.0, 10.0], 50) + rng_new.normal(0, 0.4, 100)
X_new = np.column_stack([radius_new * np.cos(angle_new), radius_new * np.sin(angle_new)])
road_new = np.repeat([0, 1], 50)
new_scores = transform_new(X_new, X, K, lam, A, gamma)

s0, s1 = scores[road == 0, 0], scores[road == 1, 0]
outer_is_above = s1.mean() > s0.mean()
cut = (s0.max() + s1.min()) / 2 if outer_is_above else (s1.max() + s0.min()) / 2
pred_new = ((new_scores[:, 0] > cut) == outer_is_above).astype(int)
acc_new = float(np.mean(pred_new == road_new))
print(f"100 new stops placed on the right road by one threshold on component 1: {acc_new:.2f}")
assert acc_new >= 0.98, "the threshold learned on the training stops should place nearly every new stop"

plt.figure(figsize=(5, 3.5))
plt.scatter(scores[:, 0], scores[:, 1], c=road, cmap="coolwarm", s=8, alpha=0.3, label="training stops")
plt.scatter(new_scores[:, 0], new_scores[:, 1], c=road_new, cmap="coolwarm", s=30, edgecolor="black", label="new stops")
plt.axvline(cut, ls="--", color="gray")
plt.xlabel("kernel component 1")
plt.ylabel("kernel component 2")
plt.legend()
plt.show()
print("Part 7 passed")

## Solution

### Part 1: Every distance at once

In [ ]:
def squared_distances(A, B):
    """Matrix of squared Euclidean distances between the rows of A and the rows of B."""
    diff = A[:, None, :] - B[None, :, :]
    return (diff ** 2).sum(axis=-1)

### Part 2: Can one number tell the roads apart?

In [ ]:
def threshold_accuracy(values, labels):
    """Accuracy of the best single threshold on `values`, in either direction."""
    best = 0.0
    for cut in values:
        above = (values > cut).astype(int)
        acc = np.mean(above == labels)
        best = max(best, acc, 1 - acc)
    return float(best)

### Part 3: Similarities with a bandwidth

In [ ]:
def neighbor_probabilities(d_sq, sigma):
    """Gaussian similarities to the other points, normalized to a probability distribution."""
    w = np.exp(-d_sq / (2 * sigma ** 2))
    return w / w.sum()


def perplexity(p):
    """2 to the entropy of p in bits: the effective number of neighbors."""
    nz = p[p > 0]
    return float(2 ** (-(nz * np.log2(nz)).sum()))

### Part 4: The kernel matrix, centered

In [ ]:
def rbf_kernel(D_sq, gamma):
    """The Gaussian kernel exp(-gamma * squared distance), entry by entry."""
    return np.exp(-gamma * D_sq)


def center_kernel(K):
    """K with its column means and row means subtracted and the overall mean added back."""
    return K - K.mean(axis=0) - K.mean(axis=1, keepdims=True) + K.mean()

### Part 5: The top eigenvectors of the kernel

In [ ]:
def kernel_pca(Kc, k):
    """(lam, A, scores): the top k eigenpairs of the centered kernel and the training scores."""
    vals, vecs = np.linalg.eigh(Kc)
    order = np.argsort(vals)[::-1][:k]
    lam, A = vals[order], vecs[:, order]
    return lam, A, A * np.sqrt(lam)

### Part 6: The window of bandwidths

In [ ]:
def kernel_pca_scores(X, gamma, k):
    """Kernel PCA scores of the rows of X with a Gaussian kernel of bandwidth gamma."""
    Kc = center_kernel(rbf_kernel(squared_distances(X, X), gamma))
    return kernel_pca(Kc, k)[2]

### Part 7: A stop the model has never seen

In [ ]:
def transform_new(X_new, X_train, K_train, lam, A, gamma):
    """Scores of new points on the components found from X_train."""
    K_new = rbf_kernel(squared_distances(X_new, X_train), gamma)
    K_new_c = K_new - K_train.mean(axis=0) - K_new.mean(axis=1, keepdims=True) + K_train.mean()
    return K_new_c @ A / np.sqrt(lam)